# Exploring a SNOMED-CT uk extension Release

In [1]:
import json
import re

import numpy as np
import pandas as pd

## Loading the SNOMED UK extention release files

In [2]:
snomed_dir = "/home/asem/GP/ehr-data/SNOMED-CT"

Use Snapshot, instead of Full, here, as Full contains all historical concepts since 2014. Delta only contains differences from last version.
https://confluence.ihtsdotools.org/display/DOCGLOSS/Snapshot+release

In [3]:
release = "39.4.0"
date = "20250115"

base_term = f"{snomed_dir}/uk_sct2mo_{release}_{date}000001Z"
terminology = f"{base_term}/SnomedCT_MonolithRF2_PRODUCTION_{date}T120000Z/Snapshot/Terminology"

## SNOMED CT Design

### SNOMED CT Components
SNOMED CT is a clinical terminology containing concepts with unique meanings and formal logic based definitions organised into hierarchies.
For further information please see: https://confluence.ihtsdotools.org/display/DOCSTART/4.+SNOMED+CT+Basics

SNOMED CT content is represented into 3 main types of components:
- __Concepts__ representing clinical meanings that are organised into hierarchies.
- __Descriptions__ which link appropriate human readable terms to concepts
- __Relationships__ which link each concept to other related concepts

__NOTE:__ SNOMED-CT (UK Ed.) is an extension to the Int Ed. Both sets of files (Int. and the UK Ext.) are released as part of one 'UK Release'.

Load and merge the active concept from the international and UK Extention __Concept snapshot__ files

#### __Table 4.2.1-1:__ Concept file - Detailed Specification

|Field|Data type|Purpose|Mutable|Part of Primary Key|
|:-----|:-----|:-----|:-----|:-----|
|id|SCTID|Uniquely Idenfies the concept|NO|YES (Full/Snapshot)|
|effectiveTime|Time|Specifies the inclusive date at which the component version's state became the then current valid state of the component.|YES|YES (Full)<br>Optional (Snapshot)|
|active|Boolean|Specifies whether the concept was active or inactive from the nominal release date specified by the effectiveTime.|YES|NO|
|moduleId|SCTID|Identifies the concept version's module. Set to a descendant of 900000000000443000(Module) within the metadata hierarchy.|YES|NO|
|definitionStatusId|SCTID|Specifies if the concept version is primitive or defined. Set to a descendant of 900000000000444006(Definition status)in the metadata hierarchy.|YES|NO|

Taken from: https://confluence.ihtsdotools.org/display/DOCRELFMT

In [6]:
terms = parse_file(f"{terminology}/sct2_Concept_MONOSnapshot_GB_{date}.txt")
active_terms = terms[terms.active == "1"]  # active concepts are represented with 1
inactive_terms = terms[terms.active != "1"]

In [7]:
# Every concept has a unique concept identifier: active_terms['id']
active_terms.describe()

,id,effectiveTime,active,moduleId,definitionStatusId
count,808816,808816,808816,808816,808816
unique,808816,302,1,7,2
top,999003181000000104,20020131,1,999000011000001104,900000000000074008
freq,1,151196,808816,404474,660593


In [8]:
inactive_terms.describe()

,id,effectiveTime,active,moduleId,definitionStatusId
count,307625,307625,307625,307625,307625
unique,307625,297,1,6,2
top,999004541000000101,20020131,0,900000000000207008,900000000000074008
freq,1,47555,307625,144642,307615


Load and merge the active descriptions from the international and UK Extention __Description snapshot__ files

#### __Table 4.2.2-1:__ Description file - Detailed Specification

|Field|Data type|Purpose|Mutable|Part of Primary Key|
|:-----|:-----|:-----|:-----|:-----|
|id|SCTID|Uniquely identifies the description.|NO|YES (Full/Snapshot)|
|effectiveTime|Time|Specifies the inclusive date at which the component version's state became the then current valid state of the component|YES|YES (Full)<br>Optional \|Snapshot\||
|active|Boolean|Specifies whether the state of the description was active or inactive from the nominal release date specified by the effectiveTime.|YES|NO|
|moduleId|SCTID|Identifies the description version's module. Set to a child of 900000000000443000\|Module\| within the metadata hierarchy.|YES|NO|
|conceptId|SCTID|Identifies the concept to which this description applies. Set to the identifier of a concept in the 138875005 \|SNOMED CT Concept\| hierarchy within the Concept. Note that a specific version of a description is not directly bound to a specific version of the concept to which it applies. Which version of a description applies to a concept depends on its effectiveTime and the point in time at which it is accessed.|NO|NO|
|languageCode|String|Specifies the language of the description text using the two character ISO-639-1 code. Note that this specifies a language level only, not a dialect or country code.|NO|NO|
|typeId|SCTID|Identifies whether the description is fully specified name a synonym or other description type. This field is set to a child of 900000000000446008\|Description type\| in the Metadata hierarchy.|NO|NO|
|term|String|The description version's text value, represented in UTF-8 encoding.|YES|NO|
|caseSignificanceId|SCTID|Identifies the concept enumeration value that represents the case significance of this description version. For example, the term may be completely case sensitive, case insensitive or initial letter case insensitive. This field will be set to a child of 900000000000447004\|Case significance\| within the metadata hierarchy.|YES|NO|

Taken from: https://confluence.ihtsdotools.org/display/DOCRELFMT

In [9]:
desc = parse_file(f"{terminology}/sct2_Description_MONOSnapshot-en_GB_{date}.txt")
active_descs = desc[desc.active == "1"]
inactive_descs = desc[desc.active != "1"]

In [10]:
active_descs.head()

,id,effectiveTime,active,moduleId,conceptId,languageCode,typeId,term,caseSignificanceId
0,101013,20170731,1,900000000000207008,126813005,en,900000000000013009,Neoplasm of anterior aspect of epiglottis,900000000000448009
1,102018,20170731,1,900000000000207008,126814004,en,900000000000013009,Neoplasm of junctional region of epiglottis,900000000000448009
2,103011,20170731,1,900000000000207008,126815003,en,900000000000013009,Neoplasm of lateral wall of oropharynx,900000000000448009
3,104017,20170731,1,900000000000207008,126816002,en,900000000000013009,Neoplasm of posterior wall of oropharynx,900000000000448009
4,105016,20170731,1,900000000000207008,126817006,en,900000000000013009,Neoplasm of esophagus,900000000000448009


In [11]:
print(inactive_descs.conceptId.nunique(), inactive_terms.id.nunique())

356598 307625


Load and merge the relationships from the international and UK Extention __Relationship snapshot__ files

#### __Table 4.2.3-1:__ Relationship file - Detailed specification

|Field|Data type|Purpose|Mutable|Part of Primary Key|
|:-----|:-----|:-----|:-----|:-----|
|id|SCTID|Uniquely identifies the relationship.|NO|YES(Full/Snapshot)|
|effectiveTime|Time|Specifies the inclusive date at which the component version's state became the then current valid state of the component.|YES|YES(Full) Optional(Snapshot)|
|active|Boolean|Specifies whether the state of the relationship was active or inactive from the nominal release date specified by the effectiveTime field.|YES|NO|
|moduleId|SCTID|Identifies the relationship version's module. Set to a child of 900000000000443000\|Module\| within the metadata hierarchy.|YES|NO|
|sourceId|SCTID|Identifies the source concept of the relationship version. That is the concept defined by this relationship. Set to the identifier of a concept.|NO|NO|
|destinationId|SCTID|Identifies the concept that is the destination of the relationship version.<br>That is the concept representing the value of the attribute represented by the typeId column.<br>Set to the identifier of a concept.<br>Note that the values that can be applied to particular attributes are formally defined by the SNOMED CT Machine Readable Concept Model.|NO|NO|
|relationshipGroup|Integer|Groups together relationship versions that are part of a logically associated relationshipGroup. All active Relationship records with the same relationshipGroup number and sourceId are grouped in this way.|YES|NO|
|typeId|SCTID|Identifies the concept that represent the defining attribute (or relationship type) represented by this relationship version.<br><br>That is the concept representing the value of the attribute represented by the typeId column. <br><br>Set to the identifier of a concept. The concept identified must be either 116680003\|Is a\| or a subtype of 410662002\|Concept model attribute\|. The concepts that can be used as in the typeId column are formally defined as follows:<br>116680003\|is a\| OR < 410662002\|concept model attribute\|<br><br>__Note__ that the attributes that can be applied to particular concepts are formally defined by the SNOMED CT Machine Readable Concept Model.|NO|NO|
|characteristicTypeId|SCTID|A concept enumeration value that identifies the characteristic type of the relationship version (i.e. whether the relationship version is defining, qualifying, etc.) This field is set to a descendant of 900000000000449001\|Characteristic type\|in the metadata hierarchy.|YES|NO|
|modifierId|SCTID|A concept enumeration value that identifies the type of Description Logic(DL) restriction (some, all, etc.). Set to a child of 900000000000450001\|Modifier\| in the metadata hierarchy.<br> __Note__ Currently the only value used in this column is 900000000000451002\|Some\| and thus in practical terms this column can be ignored.|YES|NO|

Taken from: https://confluence.ihtsdotools.org/display/DOCRELFMT

## SNOMED CT Concept Model

<img src="img/Association Between Files from 2019.png">

Taken from: https://confluence.ihtsdotools.org/display/DOCRELFMT

Find the fully specified name, Synonym or Definition of a SNOMED concept

__Description type__

|Type id|Term|
|:---:|:---|
|900000000000003001|Fully specified name|
|900000000000013009|Synonym|
|900000000000550004|Definition|


Create a DataFrame which contains only the active SNOMED codes and their fully specified name

In [12]:
code_with_desc = pd.merge(
    terms, desc[desc["typeId"] == "900000000000003001"], left_on=["id"], right_on=["conceptId"], how="inner"
)
active_with_desc = pd.merge(
    active_terms,
    active_descs[active_descs["typeId"] == "900000000000003001"],
    left_on=["id"],
    right_on=["conceptId"],
    how="inner",
)
inactive_with_desc = pd.merge(
    inactive_terms,
    inactive_descs[inactive_descs["typeId"] == "900000000000003001"],
    left_on=["id"],
    right_on=["conceptId"],
    how="inner",
)
code_with_desc.describe()

,id_x,effectiveTime_x,active_x,moduleId_x,definitionStatusId,id_y,effectiveTime_y,active_y,moduleId_y,conceptId,languageCode,typeId,term,caseSignificanceId
count,1402099,1402099,1402099,1402099,1402099,1402099,1402099,1402099,1402099,1402099,1402099,1402099,1402099,1402099
unique,1116441,302,2,7,2,1402099,302,2,7,1116441,1,1,1292249,3
top,20968811000001109,20020131,1,900000000000207008,900000000000074008,3006825011,20170731,1,900000000000207008,20968811000001109,en,900000000000003001,Retired procedure,900000000000017005
freq,8,281079,998960,677493,1207424,1,300922,1118900,679522,8,1402099,1402099,4042,581226


In [13]:
active_with_desc.describe()

,id_x,effectiveTime_x,active_x,moduleId_x,definitionStatusId,id_y,effectiveTime_y,active_y,moduleId_y,conceptId,languageCode,typeId,term,caseSignificanceId
count,811269,811269,811269,811269,811269,811269,811269,811269,811269,811269,811269,811269,811269,811269
unique,808816,302,1,7,2,811269,301,1,7,808816,1,1,811269,3
top,776625003,20020131,1,999000011000001104,900000000000074008,3006825011,20170731,1,999000011000001104,776625003,en,900000000000003001,Nocardia niwae (organism),900000000000017005
freq,2,151233,811269,404474,660710,1,212708,811269,404474,2,811269,811269,1,397583


In [14]:
inactive_with_desc.describe()

,id_x,effectiveTime_x,active_x,moduleId_x,definitionStatusId,id_y,effectiveTime_y,active_y,moduleId_y,conceptId,languageCode,typeId,term,caseSignificanceId
count,95508,95508,95508,95508,95508,95508,95508,95508,95508,95508,95508,95508,95508,95508
unique,90928,198,1,6,2,95508,171,1,6,90928,1,1,76710,3
top,323886004,20020131,0,900000000000207008,900000000000074008,999003731000000115,20080731,0,900000000000207008,323886004,en,900000000000003001,Retired procedure,900000000000020002
freq,6,47283,95508,80374,95497,1,50247,95508,80624,6,95508,95508,4042,63624


Occasionally within UK SNOMED ed. releases there are concepts which have >1 active primary descriptions. These extra descriptions need to be removed.


In [15]:
# Inspect the duplicates
active_with_desc[active_with_desc.duplicated(["id_x"], keep="first")]

,id_x,effectiveTime_x,active_x,moduleId_x,definitionStatusId,id_y,effectiveTime_y,active_y,moduleId_y,conceptId,languageCode,typeId,term,caseSignificanceId
11350,14691008,20020131,1,900000000000207008,900000000000074008,3509191000001114,20250115,1,999000041000000102,14691008,en,900000000000003001,Yttrium [Y-90] (substance),900000000000017005
11413,14767006,20020131,1,900000000000207008,900000000000074008,497031000001117,20240731,1,999000041000000102,14767006,en,900000000000003001,Human alpha1-proteinase inhibitor (substance),900000000000017005
19512,25130003,20020131,1,900000000000207008,900000000000074008,2309051000001113,20220706,1,999000041000000102,25130003,en,900000000000003001,Dimethoxymethane (substance),900000000000017005
20639,26575001,20020131,1,900000000000207008,900000000000074008,2309021000001117,20220706,1,999000041000000102,26575001,en,900000000000003001,Pimenta racemosa oil (substance),900000000000017005
22725,29218008,20020131,1,900000000000207008,900000000000074008,2464771000001112,20221026,1,999000041000000102,29218008,en,900000000000003001,Indium [In-111] pentetate (DTPA) (substance),900000000000017005
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
362344,58811000052103,20190731,1,900000000000207008,900000000000074008,1116841000001118,20210929,1,999000041000000102,58811000052103,en,900000000000003001,Peritumoral (qualifier value),900000000000017005
362349,58831000052108,20190731,1,900000000000207008,900000000000074008,1116761000001112,20210929,1,999000041000000102,58831000052108,en,900000000000003001,Subretinal (qualifier value),900000000000017005
362898,66621000052103,20210131,1,900000000000207008,900000000000074008,1939921000001116,20220216,1,999000041000000102,66621000052103,en,900000000000003001,Sublabial (qualifier value),900000000000017005
394372,1001081000202104,20240701,1,900000000000207008,900000000000073002,10617091000202113,20240701,1,900000000000207008,1001081000202104,en,900000000000003001,Product containing only trofosfamide (medicina...,900000000000448009


In [16]:
code_with_desc[code_with_desc.duplicated(["id_x"], keep="first")]

,id_x,effectiveTime_x,active_x,moduleId_x,definitionStatusId,id_y,effectiveTime_y,active_y,moduleId_y,conceptId,languageCode,typeId,term,caseSignificanceId
1,100005,20020131,0,900000000000207008,900000000000074008,2709997016,20080731,1,900000000000207008,100005,en,900000000000003001,SNOMED RT Concept (special concept),900000000000017005
27,125001,20020131,1,900000000000207008,900000000000074008,3650088017,20180731,1,900000000000207008,125001,en,900000000000003001,Ferrous (59-Fe) sulfate (substance),900000000000020002
30,127009,20210131,1,900000000000207008,900000000000073002,2966522015,20170731,1,900000000000207008,127009,en,900000000000003001,Miscarriage with laceration of cervix (disorder),900000000000448009
41,137004,20020131,0,900000000000207008,900000000000074008,2718270017,20170731,1,900000000000207008,137004,en,900000000000003001,Retired procedure (procedure),900000000000448009
48,143002,20020131,0,900000000000207008,900000000000074008,2711722019,20170731,1,900000000000207008,143002,en,900000000000003001,Retired procedure (procedure),900000000000448009
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1401919,999001551000000100,20200930,0,999000021000000109,900000000000074008,999005811000000119,20190601,1,999000021000000109,999001551000000100,en,900000000000003001,Mobility issues simple reference set (foundati...,900000000000020002
1401926,999001611000000100,20180401,0,999000011000000103,900000000000074008,999003731000000115,20150401,0,999000011000000103,999001611000000100,en,900000000000003001,Skin care and wound management simple referenc...,900000000000020002
1401964,999001981000000101,20190601,1,999000021000000109,900000000000074008,999004551000000119,20230215,0,999000021000000109,999001981000000101,en,900000000000003001,Resuscitation decision findings simple referen...,900000000000020002
1401979,999002121000000109,20190601,1,999000021000000109,900000000000074008,999004861000000118,20190601,0,999000021000000109,999002121000000109,en,900000000000003001,Accessible information - requires communicatio...,900000000000020002


In [17]:
inactive_with_desc[inactive_with_desc.duplicated(["id_x"], keep="first")]

,id_x,effectiveTime_x,active_x,moduleId_x,definitionStatusId,id_y,effectiveTime_y,active_y,moduleId_y,conceptId,languageCode,typeId,term,caseSignificanceId
26,353008,20190131,0,900000000000207008,900000000000074008,1457572016,20060731,0,900000000000207008,353008,en,900000000000003001,IV/irrigation monitoring (regime/therapy),900000000000017005
27,353008,20190131,0,900000000000207008,900000000000074008,2612266016,20140131,0,900000000000207008,353008,en,900000000000003001,Intravenous (IV)/irrigation monitoring (regime...,900000000000020002
147,1552008,20210131,0,900000000000207008,900000000000074008,1186518014,20190131,0,900000000000207008,1552008,en,900000000000003001,Saluretic drug (product),900000000000448009
214,2092003,20210930,0,900000000000207008,900000000000074008,1186858016,20030731,0,900000000000207008,2092003,en,900000000000003001,"Malignant melanoma, morphology (morphologic ab...",900000000000020002
215,2092003,20210930,0,900000000000207008,900000000000074008,1762747019,20140131,0,900000000000207008,2092003,en,900000000000003001,"Malignant melanoma, no ICD-O subtype (morpholo...",900000000000020002
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95403,35845211000001106,20181128,0,999000011000001104,900000000000074008,3554971000001116,20240925,0,999000011000001104,35845211000001106,en,900000000000003001,Emulsifying ointment (Ennogen Healthcare Ltd) ...,900000000000017005
95404,35845211000001106,20181128,0,999000011000001104,900000000000074008,110152501000001113,20240410,0,999000011000001104,35845211000001106,en,900000000000003001,Ovelle emulsifying ointment (Ovelle Pharmaceut...,900000000000017005
95433,39026411000001108,20221123,0,999000011000001104,900000000000074008,2445571000001112,20230510,0,999000011000001104,39026411000001108,en,900000000000003001,Juvela gluten free pizza base (Hero UK Limited...,900000000000017005
95434,39026411000001108,20221123,0,999000011000001104,900000000000074008,2617551000001116,20230510,0,999000011000001104,39026411000001108,en,900000000000003001,Juvela gluten free pizza base (Juvela Ltd) 300...,900000000000017005


In [18]:
# drop duplicates
active_with_desc = active_with_desc.drop_duplicates(["id_x"], keep="first")
assert len(active_with_desc) == len(active_terms)

inactive_with_desc = inactive_with_desc.drop_duplicates(["id_x"], keep="first")

code_with_desc = code_with_desc.drop_duplicates(["id_x"], keep="first")

Create the top-level Concept which each concept is linked to:
tui -> term unique identifier

In [19]:
def find_tui(concept_name):
    return re.match("\((\w+\s?.?\s?\w+.?\w+.?\w+.?)\)$")


active_with_desc["tui"] = active_with_desc["term"].str.extract("\((\w+\s?.?\s?\w+.?\w+.?\w+.?)\)$")
inactive_with_desc["tui"] = inactive_with_desc["term"].str.extract("\((\w+\s?.?\s?\w+.?\w+.?\w+.?)\)$")
code_with_desc["tui"] = code_with_desc["term"].str.extract("\((\w+\s?.?\s?\w+.?\w+.?\w+.?)\)$")

<>:2: SyntaxWarning: invalid escape sequence '\('
<>:3: SyntaxWarning: invalid escape sequence '\('
<>:4: SyntaxWarning: invalid escape sequence '\('
<>:5: SyntaxWarning: invalid escape sequence '\('
<>:2: SyntaxWarning: invalid escape sequence '\('
<>:3: SyntaxWarning: invalid escape sequence '\('
<>:4: SyntaxWarning: invalid escape sequence '\('
<>:5: SyntaxWarning: invalid escape sequence '\('
/tmp/ipykernel_225306/749618863.py:2: SyntaxWarning: invalid escape sequence '\('
  return re.match("\((\w+\s?.?\s?\w+.?\w+.?\w+.?)\)$")
/tmp/ipykernel_225306/749618863.py:3: SyntaxWarning: invalid escape sequence '\('
  active_with_desc['tui'] = active_with_desc['term'].str.extract("\((\w+\s?.?\s?\w+.?\w+.?\w+.?)\)$")
/tmp/ipykernel_225306/749618863.py:4: SyntaxWarning: invalid escape sequence '\('
  inactive_with_desc['tui'] = inactive_with_desc['term'].str.extract("\((\w+\s?.?\s?\w+.?\w+.?\w+.?)\)$")
/tmp/ipykernel_225306/749618863.py:5: SyntaxWarning: invalid escape sequence '\('
  code_wi

In [20]:
active_with_desc.describe()

,id_x,effectiveTime_x,active_x,moduleId_x,definitionStatusId,id_y,effectiveTime_y,active_y,moduleId_y,conceptId,languageCode,typeId,term,caseSignificanceId,tui
count,808816,808816,808816,808816,808816,808816,808816,808816,808816,808816,808816,808816,808816,808816,808816
unique,808816,302,1,7,2,808816,301,1,7,808816,1,1,808816,3,57
top,999003181000000104,20020131,1,999000011000001104,900000000000074008,999007161000000115,20170731,1,999000011000001104,999003181000000104,en,900000000000003001,National Cancer Registration and Analysis Serv...,900000000000017005,physical object
freq,1,151196,808816,404474,660593,1,212708,808816,404474,1,808816,808816,1,395134,214243


In [21]:
active_with_desc[active_with_desc["tui"].isnull()].values

array([], shape=(0, 15), dtype=object)

In [22]:
# The number of unique TUIs
active_with_desc["tui"].unique()

array(['organism', 'substance', 'procedure', 'body structure', 'disorder',
       'occupation', 'finding', 'qualifier value',
       'morphologic abnormality', 'cell structure', 'physical object',
       'regime/therapy', 'product', 'medicinal product', 'cell', 'person',
       'environment', 'observable entity', 'event', 'religion/philosophy',
       'attribute', 'situation', 'medicinal product form',
       'navigational concept', 'physical force', 'clinical drug',
       'social concept', 'tumor staging', 'specimen', 'basic dose form',
       'dose form', 'linkage concept', 'staging scale', 'record artifact',
       'assessment scale', 'SNOMED RT+CTV3', 'geographic location',
       'environment / location', 'special concept', 'namespace concept',
       'ethnic group', 'racial group', 'link assertion',
       'foundation metadata concept', 'core metadata concept',
       'disposition', 'unit of presentation', 'OWL metadata concept',
       'state of matter', 'transformation', 'inte

Explore what each tui contains:

In [23]:
active_with_desc[active_with_desc["tui"] == "number"]

,id_x,effectiveTime_x,active_x,moduleId_x,definitionStatusId,id_y,effectiveTime_y,active_y,moduleId_y,conceptId,languageCode,typeId,term,caseSignificanceId,tui


### Create the input required for a MedCAT concept database

In [24]:
snomed_cdb_active_only = active_with_desc[["id_x", "term", "tui"]]
snomed_cdb_inactive_only = inactive_with_desc[["id_x", "term", "tui"]]
snomed_cdb_active_only.columns = ["cui", "str", "sty"]
snomed_cdb_inactive_only.columns = ["cui", "str", "sty"]
snomed_cdb_all = code_with_desc[["id_x", "term", "tui"]]
snomed_cdb_all.columns = ["cui", "str", "sty"]

snomed_cdb_active_only["cui"] = snomed_cdb_active_only.cui.apply(lambda code: f"S-{code}")
snomed_cdb_active_only["onto"] = "SNOMED-CT"

snomed_cdb_inactive_only["cui"] = snomed_cdb_inactive_only.cui.apply(lambda code: f"S-{code}")
snomed_cdb_inactive_only["onto"] = "SNOMED-CT"


snomed_cdb_all["cui"] = snomed_cdb_all.cui.apply(lambda code: f"S-{code}")
snomed_cdb_all["onto"] = "SNOMED-CT"

/tmp/ipykernel_225306/2402534566.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  snomed_cdb_active_only['cui'] = snomed_cdb_active_only.cui.apply(lambda code: f'S-{code}')
/tmp/ipykernel_225306/2402534566.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  snomed_cdb_active_only['onto'] = 'SNOMED-CT'
/tmp/ipykernel_225306/2402534566.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats i

In [25]:
snomed_cdb_active_only["tty"] = np.nan
snomed_cdb_active_only["tui"] = np.nan

snomed_cdb_inactive_only["tty"] = np.nan
snomed_cdb_inactive_only["tui"] = np.nan

snomed_cdb_all["tty"] = np.nan
snomed_cdb_all["tui"] = np.nan

/tmp/ipykernel_225306/3794539430.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  snomed_cdb_active_only['tty'] = np.nan
/tmp/ipykernel_225306/3794539430.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  snomed_cdb_active_only['tui'] = np.nan
/tmp/ipykernel_225306/3794539430.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/

In [26]:
snomed_cdb_active_only  # just for active concepts

,cui,str,sty,onto,tty,tui
0,S-101009,Quilonia ethiopica (organism),organism,SNOMED-CT,NaN,NaN
1,S-102002,Hemoglobin Okaloosa (substance),substance,SNOMED-CT,NaN,NaN
2,S-103007,Squirrel fibroma virus (organism),organism,SNOMED-CT,NaN,NaN
3,S-104001,Excision of lesion of patella (procedure),procedure,SNOMED-CT,NaN,NaN
4,S-107008,Structure of fetal part of placenta (body stru...,body structure,SNOMED-CT,NaN,NaN
...,...,...,...,...,...,...
811264,S-999004501000000104,Routine childhood immunisation schedule proced...,foundation metadata concept,SNOMED-CT,NaN,NaN
811265,S-999004521000000108,Health issue severity simple reference set (fo...,foundation metadata concept,SNOMED-CT,NaN,NaN
811266,S-999004531000000105,Health issue certainty simple reference set (f...,foundation metadata concept,SNOMED-CT,NaN,NaN
811267,S-999480551000087103,Aspergillus japonicus (organism),organism,SNOMED-CT,NaN,NaN


In [27]:
snomed_cdb_inactive_only

,cui,str,sty,onto,tty,tui
0,S-100005,SNOMED RT Concept,NaN,SNOMED-CT,NaN,NaN
1,S-137004,Retired procedure,NaN,SNOMED-CT,NaN,NaN
2,S-143002,Retired procedure,NaN,SNOMED-CT,NaN,NaN
3,S-152006,Retired procedure,NaN,SNOMED-CT,NaN,NaN
4,S-160007,Removal of foreign body of tendon AND/OR tendo...,procedure,SNOMED-CT,NaN,NaN
...,...,...,...,...,...,...
95503,S-42924711000001106,Recordati UK Ltd (qualifier value),qualifier value,SNOMED-CT,NaN,NaN
95504,S-999000821000001101,Enhanced Services - Seasonal Flu vaccines simp...,foundation metadata concept,SNOMED-CT,NaN,NaN
95505,S-999001411000000103,Assessment type simple reference set (foundati...,foundation metadata concept,SNOMED-CT,NaN,NaN
95506,S-999001551000000100,Mobility finding simple reference set (foundat...,foundation metadata concept,SNOMED-CT,NaN,NaN


In [28]:
snomed_cdb_all

,cui,str,sty,onto,tty,tui
0,S-100005,SNOMED RT Concept,NaN,SNOMED-CT,NaN,NaN
2,S-101009,Quilonia ethiopica (organism),organism,SNOMED-CT,NaN,NaN
3,S-102002,Hemoglobin Okaloosa (substance),substance,SNOMED-CT,NaN,NaN
4,S-103007,Squirrel fibroma virus (organism),organism,SNOMED-CT,NaN,NaN
5,S-104001,Excision of lesion of patella (procedure),procedure,SNOMED-CT,NaN,NaN
...,...,...,...,...,...,...
1402094,S-999004521000000108,Health issue severity simple reference set (fo...,foundation metadata concept,SNOMED-CT,NaN,NaN
1402095,S-999004531000000105,Health issue certainty simple reference set (f...,foundation metadata concept,SNOMED-CT,NaN,NaN
1402096,S-999004541000000101,Unified Test List result observables simple re...,foundation metadata concept,SNOMED-CT,NaN,NaN
1402097,S-999480551000087103,Aspergillus japonicus (organism),organism,SNOMED-CT,NaN,NaN


#### Create a MedCAT concept database including all synonyms

In [29]:
_ = pd.merge(active_terms, active_descs, left_on=["id"], right_on=["conceptId"], how="inner")
active_with_primary_desc = _[_["typeId"] == "900000000000003001"]
active_with_primary_desc = active_with_primary_desc.drop_duplicates(["id_x"], keep="first")
active_with_synonym_desc = _[_["typeId"] == "900000000000013009"]
active_with_all_desc = pd.concat([active_with_primary_desc, active_with_synonym_desc])

In [30]:
_ = pd.merge(inactive_terms, inactive_descs, left_on=["id"], right_on=["conceptId"], how="inner")
inactive_with_primary_desc = _[_["typeId"] == "900000000000003001"]
inactive_with_primary_desc = inactive_with_primary_desc.drop_duplicates(["id_x"], keep="first")
inactive_with_synonym_desc = _[_["typeId"] == "900000000000013009"]
inactive_with_all_desc = pd.concat([inactive_with_primary_desc, inactive_with_synonym_desc])

In [31]:
_ = pd.merge(terms, desc, left_on=["id"], right_on=["conceptId"], how="inner")
code_with_primary_desc = _[_["typeId"] == "900000000000003001"]
code_with_primary_desc = code_with_primary_desc.drop_duplicates(["id_x"], keep="first")
code_with_synonym_desc = _[_["typeId"] == "900000000000013009"]
code_with_all_desc = pd.concat([code_with_primary_desc, code_with_synonym_desc])

In [32]:
# Check if there are the same amount of active concepts
assert len(active_with_all_desc[active_with_all_desc["typeId"] == "900000000000003001"]) == len(active_terms)

In [33]:
snomed_cdb_active_df = pd.merge(
    active_with_all_desc, active_with_desc, left_on=["id_x"], right_on=["conceptId"], how="inner"
)
snomed_cdb_inactive_df = pd.merge(
    inactive_with_all_desc, inactive_with_desc, left_on=["id_x"], right_on=["conceptId"], how="inner"
)
snomed_cdb_all_df = pd.merge(code_with_all_desc, code_with_desc, left_on=["id_x"], right_on=["conceptId"], how="inner")

In [34]:
# clean up the merge and rename the columns to fit the medcat Concept database criteria
snomed_cdb_active_df = snomed_cdb_active_df.loc[:, ["id_x_x", "term_x", "typeId_x", "tui"]]
snomed_cdb_active_df.columns = ["cui", "str", "tty", "sty"]
snomed_cdb_active_df["onto"] = "SNOMED-CT"
snomed_cdb_active_df["tty"] = snomed_cdb_active_df["tty"].replace(["900000000000003001", "900000000000013009"], [1, 0])
snomed_cdb_active_df["cui"] = "S-" + snomed_cdb_active_df["cui"].astype(str)
snomed_cdb_active_df

/tmp/ipykernel_225306/3578942014.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  snomed_cdb_active_df['tty'] = snomed_cdb_active_df['tty'].replace(['900000000000003001', '900000000000013009'], [1,0])


,cui,str,tty,sty,onto
0,S-101009,Quilonia ethiopica (organism),1,organism,SNOMED-CT
1,S-102002,Hemoglobin Okaloosa (substance),1,substance,SNOMED-CT
2,S-103007,Squirrel fibroma virus (organism),1,organism,SNOMED-CT
3,S-104001,Excision of lesion of patella (procedure),1,procedure,SNOMED-CT
4,S-107008,Structure of fetal part of placenta (body stru...,1,body structure,SNOMED-CT
...,...,...,...,...,...
1877196,S-999004501000000104,Routine childhood immunisation schedule proced...,0,foundation metadata concept,SNOMED-CT
1877197,S-999004521000000108,Health issue severity simple reference set,0,foundation metadata concept,SNOMED-CT
1877198,S-999004531000000105,Health issue certainty simple reference set,0,foundation metadata concept,SNOMED-CT
1877199,S-999480551000087103,Aspergillus japonicus,0,organism,SNOMED-CT


In [35]:
# clean up the merge and rename the columns to fit the medcat Concept database criteria
snomed_cdb_inactive_df = snomed_cdb_inactive_df.loc[:, ["id_x_x", "term_x", "typeId_x", "tui"]]
snomed_cdb_inactive_df.columns = ["cui", "str", "tty", "sty"]
snomed_cdb_inactive_df["onto"] = "SNOMED-CT"
snomed_cdb_inactive_df["tty"] = snomed_cdb_inactive_df["tty"].replace(
    ["900000000000003001", "900000000000013009"], [1, 0]
)
snomed_cdb_inactive_df["cui"] = "S-" + snomed_cdb_inactive_df["cui"].astype(str)
snomed_cdb_inactive_df

/tmp/ipykernel_225306/234839216.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  snomed_cdb_inactive_df['tty'] = snomed_cdb_inactive_df['tty'].replace(['900000000000003001', '900000000000013009'], [1,0])


,cui,str,tty,sty,onto
0,S-100005,SNOMED RT Concept,1,NaN,SNOMED-CT
1,S-137004,Retired procedure,1,NaN,SNOMED-CT
2,S-143002,Retired procedure,1,NaN,SNOMED-CT
3,S-152006,Retired procedure,1,NaN,SNOMED-CT
4,S-160007,Removal of foreign body of tendon AND/OR tendo...,1,procedure,SNOMED-CT
...,...,...,...,...,...
110217,S-42924711000001106,Recordati UK Ltd,0,qualifier value,SNOMED-CT
110218,S-999000821000001101,Enhanced Services - Seasonal Flu vaccines simp...,0,foundation metadata concept,SNOMED-CT
110219,S-999001411000000103,Assessment type simple reference set,0,foundation metadata concept,SNOMED-CT
110220,S-999001551000000100,Mobility finding simple reference set,0,foundation metadata concept,SNOMED-CT


In [36]:
# clean up the merge and rename the columns to fit the medcat Concept database criteria
snomed_cdb_all_df = snomed_cdb_all_df.loc[:, ["id_x_x", "term_x", "typeId_x", "tui"]]
snomed_cdb_all_df.columns = ["cui", "str", "tty", "sty"]
snomed_cdb_all_df["onto"] = "SNOMED-CT"
snomed_cdb_all_df["tty"] = snomed_cdb_all_df["tty"].replace(["900000000000003001", "900000000000013009"], [1, 0])
snomed_cdb_all_df["cui"] = "S-" + snomed_cdb_all_df["cui"].astype(str)
snomed_cdb_all_df

/tmp/ipykernel_225306/1179039197.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  snomed_cdb_all_df['tty'] = snomed_cdb_all_df['tty'].replace(['900000000000003001', '900000000000013009'], [1,0])


,cui,str,tty,sty,onto
0,S-100005,SNOMED RT Concept,1,NaN,SNOMED-CT
1,S-101009,Quilonia ethiopica (organism),1,organism,SNOMED-CT
2,S-102002,Hemoglobin Okaloosa (substance),1,substance,SNOMED-CT
3,S-103007,Squirrel fibroma virus (organism),1,organism,SNOMED-CT
4,S-104001,Excision of lesion of patella (procedure),1,procedure,SNOMED-CT
...,...,...,...,...,...
2938414,S-999004531000000105,Health issue certainty simple reference set,0,foundation metadata concept,SNOMED-CT
2938415,S-999004541000000101,UTL (Unified Test List) result observables sim...,0,foundation metadata concept,SNOMED-CT
2938416,S-999004541000000101,Unified Test List result observables simple re...,0,foundation metadata concept,SNOMED-CT
2938417,S-999480551000087103,Aspergillus japonicus,0,organism,SNOMED-CT


There are 58 Semantic Tag categories total in the SNOMED taxonomy
- There is one root concept.
- There are 19 top level terms in bold.
- There are 39 sub terms.

Each semantic Tag is provided with a __term unique identifier (TUI)__ which are structured are follows:
T- {##}{1#}{2#}{3#}
- {T- }  -> Common to all codes
- {##}  -> Top level terms in alphabetical order
- {#1}  -> First level term group
- {#2}  -> Second level term group
- {#3}  -> Third level term group


### Specifying top levels terms and Semantic Tags

|Top level code|Term (TUI) |Semantic Tag|
|:---:|:---:|:---|
|__Root code__|__T-00000__|__SNOMED RT+CTV3__|
||||
|__Y__|__T-01000__|__Body structure (body structure)__|
|N|T-01100|morphologic abnormality|
|N|T-01200|cell structure|
|N|T-01210|cell|
||||
|__Y__|__T-02000__|__Clinical finding (finding)__|
|N|T-02100|disorder|
||||
|__Y__|__T-03000__|__Environment or geographical location (environment / location)__|
|N|T-03100|environment|
|N|T-03200|geographic location|
||||
|__Y__|__T-04000__|__Event (event)__|
||||
|__Y__|__T-05000__|__Observable entity (observable entity)__|
||||
|__Y__|__T-06000__|__Organism (organism)__|
||||
|__Y__|__T-07000__|__Pharmaceutical / biologic product (product)__|
|N|T-07100|medicinal product|
|N|T-07110|medicinal product form|
|N|T-07111|clinical drug|
|__Y__|__T-08000__|__Physical force (physical force)__|
||||
|__Y__|__T-09000__|__Physical object (physical object)__|
||||
|__Y__|__T-10000__|__Procedure (procedure)__|
|N|T-10100|regime/therapy|
||||
|__Y__|__T-11000__|__Qualifier value (qualifier value)__|
|N|T-11100|administration method|
|N|T-11200|disposition|
|N|T-11300|intended site|
|N|T-11010|number|
|N|T-11400|release characteristic|
|N|T-11500|transformation|
|N|T-11020|basic dose form|
|N|T-11030|dose form|
|N|T-11600|role|
|N|T-11700|state of matter|
|N|T-11040|unit of presentation|
||||
|__Y__|__T-12000__|__Record artifact (record artifact)__|
||||
|__Y__|__T-13000__|__Situation with explicit context (situation)__|
||||
|__Y__|__T-14000__|__SNOMED CT Model Component (metadata)__|
|N|T-14100|core metadata concept|
|N|T-14200|foundation metadata concept|
|N|T-14300|linkage concept|
|N|T-14310|attribute|
|N|T-14320|link assertion|
|N|T-14400|namespace concept|
|N|T-14500|OWL metadata concept|
||||
|__Y__|__T-15000__|__Social context (social concept)__|
|N|T-15100|life style|
|N|T-15010|racial group|
|N|T-15020|ethnic group|
|N|T-15200|occupation|
|N|T-15300|person|
|N|T-15400|religion/philosophy|
||||
|__Y__|__T-16000__|__Special concept (special concept)__|
|N|T-16100|inactive concept|
|N|T-16200|navigational concept|
||||
|__Y__|__T-17000__|__Specimen (specimen)__|
||||
|__Y__|__T-18000__|__Staging and scales (staging scale)__|
|N|T-18100|assessment scale|
|N|T-18200|tumor staging|
||||
|__Y__|__T-19000__|__Substance (substance)__|
||||


In [37]:
# List of all Semantic Tags
terms_list = snomed_cdb_active_df["sty"].unique().tolist()
terms_list.sort()
print(terms_list)

['OWL metadata concept', 'SNOMED RT+CTV3', 'administration method', 'assessment scale', 'attribute', 'basic dose form', 'body structure', 'cell', 'cell structure', 'clinical drug', 'core metadata concept', 'disorder', 'disposition', 'dose form', 'environment', 'environment / location', 'ethnic group', 'event', 'finding', 'foundation metadata concept', 'geographic location', 'intended site', 'link assertion', 'linkage concept', 'medicinal product', 'medicinal product form', 'metadata', 'morphologic abnormality', 'namespace concept', 'navigational concept', 'observable entity', 'occupation', 'organism', 'person', 'physical force', 'physical object', 'procedure', 'product', 'product name', 'qualifier value', 'racial group', 'record artifact', 'regime/therapy', 'release characteristic', 'religion/philosophy', 'role', 'situation', 'social concept', 'special concept', 'specimen', 'staging scale', 'state of matter', 'substance', 'supplier', 'transformation', 'tumor staging', 'unit of presenta

In [38]:
terms_dict = {
    "T-00000": "SNOMED RT+CTV3",
    "T-01000": "body structure",
    "T-01100": "morphologic abnormality",
    "T-01200": "cell structure",
    "T-01210": "cell",
    "T-02000": "finding",
    "T-02100": "disorder",
    "T-03000": "environment / location",
    "T-03100": "environment",
    "T-03200": "geographic location",
    "T-04000": "event",
    "T-05000": "observable entity",
    "T-06000": "organism",
    "T-07000": "product",
    "T-07100": "medicinal product",
    "T-07110": "medicinal product form",
    "T-07111": "clinical drug",
    "T-08000": "physical force",
    "T-09000": "physical object",
    "T-10000": "procedure",
    "T-10100": "regime/therapy",
    "T-11000": "qualifier value",
    "T-11100": "administration method",
    "T-11200": "disposition",
    "T-11300": "intended site",
    "T-11800": "supplier",
    "T-11900": "product name",
    "T-11400": "release characteristic",
    "T-11500": "transformation",
    "T-11020": "basic dose form",
    "T-11030": "dose form",
    "T-11600": "role",
    "T-11700": "state of matter",
    "T-11040": "unit of presentation",
    "T-12000": "record artifact",
    "T-13000": "situation",
    "T-14000": "metadata",
    "T-14100": "core metadata concept",
    "T-14200": "foundation metadata concept",
    "T-14300": "linkage concept",
    "T-14310": "attribute",
    "T-14320": "link assertion",
    "T-14400": "namespace concept",
    "T-14500": "OWL metadata concept",
    "T-15000": "social concept",
    "T-15100": "life style",
    "T-15010": "racial group",
    "T-15020": "ethnic group",
    "T-15200": "occupation",
    "T-15300": "person",
    "T-15400": "religion/philosophy",
    "T-16000": "special concept",
    "T-16100": "inactive concept",
    "T-16200": "navigational concept",
    "T-17000": "specimen",
    "T-18000": "staging scale",
    "T-18100": "assessment scale",
    "T-18200": "tumor staging",
    "T-19000": "substance",
}

In [39]:
# Check if all Semantic Tags are assigned a term unique identifier (TUI)
for term in terms_list:
    if term not in list(terms_dict.values()):
        print(term)

In [40]:
# # Test if the TUIs are correct for the version of snomed
# assert len(terms_list) == len(terms_dict) # check if there is the same number of groups
# for i in terms_list:
#     assert i in terms_dict.values() # check if the terms are identical

In [41]:
set(terms_dict.values()) - set(terms_list)

{'inactive concept', 'life style'}

In [42]:
# Add tui codes
dict2 = {v: k for k, v in terms_dict.items()}
snomed_cdb_active_df["tui"] = snomed_cdb_active_df["sty"].map(dict2)
snomed_cdb_active_df[["cui", "str", "onto", "tty", "tui", "sty"]]

,cui,str,onto,tty,tui,sty
0,S-101009,Quilonia ethiopica (organism),SNOMED-CT,1,T-06000,organism
1,S-102002,Hemoglobin Okaloosa (substance),SNOMED-CT,1,T-19000,substance
2,S-103007,Squirrel fibroma virus (organism),SNOMED-CT,1,T-06000,organism
3,S-104001,Excision of lesion of patella (procedure),SNOMED-CT,1,T-10000,procedure
4,S-107008,Structure of fetal part of placenta (body stru...,SNOMED-CT,1,T-01000,body structure
...,...,...,...,...,...,...
1877196,S-999004501000000104,Routine childhood immunisation schedule proced...,SNOMED-CT,0,T-14200,foundation metadata concept
1877197,S-999004521000000108,Health issue severity simple reference set,SNOMED-CT,0,T-14200,foundation metadata concept
1877198,S-999004531000000105,Health issue certainty simple reference set,SNOMED-CT,0,T-14200,foundation metadata concept
1877199,S-999480551000087103,Aspergillus japonicus,SNOMED-CT,0,T-06000,organism


In [43]:
snomed_cdb_inactive_df["tui"] = snomed_cdb_inactive_df["sty"].map(dict2)
snomed_cdb_inactive_df[["cui", "str", "onto", "tty", "tui", "sty"]]

,cui,str,onto,tty,tui,sty
0,S-100005,SNOMED RT Concept,SNOMED-CT,1,NaN,NaN
1,S-137004,Retired procedure,SNOMED-CT,1,NaN,NaN
2,S-143002,Retired procedure,SNOMED-CT,1,NaN,NaN
3,S-152006,Retired procedure,SNOMED-CT,1,NaN,NaN
4,S-160007,Removal of foreign body of tendon AND/OR tendo...,SNOMED-CT,1,T-10000,procedure
...,...,...,...,...,...,...
110217,S-42924711000001106,Recordati UK Ltd,SNOMED-CT,0,T-11000,qualifier value
110218,S-999000821000001101,Enhanced Services - Seasonal Flu vaccines simp...,SNOMED-CT,0,T-14200,foundation metadata concept
110219,S-999001411000000103,Assessment type simple reference set,SNOMED-CT,0,T-14200,foundation metadata concept
110220,S-999001551000000100,Mobility finding simple reference set,SNOMED-CT,0,T-14200,foundation metadata concept


In [44]:
snomed_cdb_all_df["tui"] = snomed_cdb_all_df["sty"].map(dict2)
snomed_cdb_all_df[["cui", "str", "onto", "tty", "tui", "sty"]]

,cui,str,onto,tty,tui,sty
0,S-100005,SNOMED RT Concept,SNOMED-CT,1,NaN,NaN
1,S-101009,Quilonia ethiopica (organism),SNOMED-CT,1,T-06000,organism
2,S-102002,Hemoglobin Okaloosa (substance),SNOMED-CT,1,T-19000,substance
3,S-103007,Squirrel fibroma virus (organism),SNOMED-CT,1,T-06000,organism
4,S-104001,Excision of lesion of patella (procedure),SNOMED-CT,1,T-10000,procedure
...,...,...,...,...,...,...
2938414,S-999004531000000105,Health issue certainty simple reference set,SNOMED-CT,0,T-14200,foundation metadata concept
2938415,S-999004541000000101,UTL (Unified Test List) result observables sim...,SNOMED-CT,0,T-14200,foundation metadata concept
2938416,S-999004541000000101,Unified Test List result observables simple re...,SNOMED-CT,0,T-14200,foundation metadata concept
2938417,S-999480551000087103,Aspergillus japonicus,SNOMED-CT,0,T-06000,organism


#### Saving your df to CSV

In [45]:
# Write the clinical terms to csv
snomed_cdb_active_df.to_csv(f"snomed_cdb_csv_SNOMEDCT_active_UK_MONORelease_{release}_{date}.csv")
snomed_cdb_inactive_df.to_csv(f"snomed_cdb_csv_SNOMEDCT_inactive_UK_MONORelease_{release}_{date}.csv")
snomed_cdb_all_df.to_csv(f"snomed_cdb_csv_SNOMEDCT_all_UK_MONORelease_{release}_{date}.csv")

In [46]:
# Test dataset for presence of COVID-19 concepts.
a = snomed_cdb_active_df[snomed_cdb_active_df["str"].str.contains("novel coronavirus")]
a

,cui,str,tty,sty,onto,tui
1341495,S-840533007,2019 novel coronavirus,0,organism,SNOMED-CT,T-06000
1341499,S-840533007,2019-nCoV (novel coronavirus),0,organism,SNOMED-CT,T-06000
1341504,S-840534001,2019 novel coronavirus antigen immunisation,0,procedure,SNOMED-CT,T-10000
1341505,S-840534001,2019 novel coronavirus antigen immunization,0,procedure,SNOMED-CT,T-10000
1341508,S-840534001,2019 novel coronavirus antigen vaccination,0,procedure,SNOMED-CT,T-10000
...,...,...,...,...,...,...
1876363,S-674814021000119106,Acute respiratory distress syndrome due to dis...,0,disorder,SNOMED-CT,T-02100
1876580,S-880529761000119102,Infection of lower respiratory tract caused by...,0,disorder,SNOMED-CT,T-02100
1876581,S-880529761000119102,Lower respiratory infection caused by 2019-nCo...,0,disorder,SNOMED-CT,T-02100
1876590,S-882784691000119100,Pneumonia caused by 2019 novel coronavirus,0,disorder,SNOMED-CT,T-02100


In [47]:
# Tuis relevant for most projects
tuisd = ["T-02000", "T-02100", "T-07000", "T-07100", "T-07110", "T-07111", "T-10000", "T-19000"]
for _ in tuisd:
    print(terms_dict[_], _)

finding T-02000
disorder T-02100
product T-07000
medicinal product T-07100
medicinal product form T-07110
clinical drug T-07111
procedure T-10000
substance T-19000


In [48]:
# To check df for covid-19 concepts
b = []
a = snomed_cdb_all_df[snomed_cdb_all_df["str"].str.contains("novel coronavirus")]
for index, _ in a[["str", "cui"]].iterrows():
    b.append(_)
# To display all:
# with pd.option_context('display.max_rows', None, 'display.max_columns', None):
#    display(b)
print(f"There are: {len(b)} concepts which contain 'novel coronavirus' ")

There are: 397 concepts which contain 'novel coronavirus' 


In [49]:
# # Functions for finding the concept name and all synonyms for a SNOMED concept

# def find_name(snomedcode):
#     """
#     Converts SNOMED code to Fully specified name and finds any Synonyms
#     """
#     df = snomed_cdb_df[(snomed_cdb_df['cui'] == snomedcode) & (snomed_cdb_df['tty'] == 1)]
#     concept_name = df['str'].values
#     return f"{''.join(concept_name)}"

# def find_syn(snomedcode):
#     """
#     Converts SNOMED code and finds all Synonyms. Not including concept name
#     """
#     df = snomed_cdb_df[(snomed_cdb_df['cui'] == snomedcode) & (snomed_cdb_df['tty'] == 0)]
#     synonym = df['str'].tolist()
#     return f"{'; '.join(synonym)}"

In [50]:
# print(find_name("S-50417007"))
# print(find_syn("S-50417007"))

## Exploring SNOMED relationships

### Root and top-level Concepts
All concepts appear from the root concept 138875005 |SNOMED CT Concept (SNOMED RT+CTV3)|


####  Table 3: Top Level Concepts 
These concepts all root from the base concept: 138875005, (SNOMED CT Concept (SNOMED RT+CTV3))<br>These concepts are all linked via the relationship typeId: 116680003, (is a)
<br>A full list of relationship types can be found as children concepts of: 106237007, (linkage concept)



|SCTID|Semantic Tag|
|:---:|:---:|
|123037004 |Body structure|
|404684003 |Clinical finding|
|272379006 |Event|
|308916002 |Environment or geographical location|
|363787002 |Observable entity|
|410607006 |Organism|
|373873005 |Pharmaceutical / biologic product|
|78621006 |Physical force|
|260787004 |Physical object|
|71388002 |Procedure|
|362981000 |Qualifier value|
|419891008 |Record artifact|
|243796009 |Situation with explicit context|
|900000000000441003 |SNOMED CT Model Component (metadata)|
|48176007 |Social context|
|370115009 |Special concept|
|123038009 |Specimen|
|254291000 |Staging and scales|
|105590001 |Substance|


Taken from Techincal implementation guide(4.1), Table 4.1-3: https://confluence.ihtsdotools.org/display/DOCTIG 

## Creating the relationship dictionaries

Parent to children structure
pt2ch = {‘\<cui_for_pt\>’, \[\<list of cuis for children\>\], …}

In [51]:
relat = parse_file(f"{terminology}/sct2_Relationship_MONOSnapshot_GB_{date}.txt")
active_relat = relat[relat.active == "1"]
inactive_relat = relat[relat.active != "1"]
active_relat.head()

,id,effectiveTime,active,moduleId,sourceId,destinationId,relationshipGroup,typeId,characteristicTypeId,modifierId
1,101021,20020131,1,900000000000207008,10000006,29857009,0,116680003,900000000000011006,900000000000451002
2,102025,20020131,1,900000000000207008,10000006,9972008,0,116680003,900000000000011006,900000000000451002
8,108026,20220706,1,999000041000000102,272026007,25087005,2,363698007,900000000000011006,900000000000451002
14,114022,20020131,1,900000000000207008,134035007,84371003,0,116680003,900000000000011006,900000000000451002
27,127021,20020131,1,900000000000207008,134136005,57250008,0,116680003,900000000000011006,900000000000451002


In [52]:
active_relat[["sourceId", "destinationId", "typeId"]] = "S-" + active_relat[
    ["sourceId", "destinationId", "typeId"]
].astype(str)

/tmp/ipykernel_225306/2612215557.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  active_relat[['sourceId','destinationId','typeId']] = 'S-' + active_relat[['sourceId','destinationId','typeId']].astype(str)


In [53]:
active_relat.head()

,id,effectiveTime,active,moduleId,sourceId,destinationId,relationshipGroup,typeId,characteristicTypeId,modifierId
1,101021,20020131,1,900000000000207008,S-10000006,S-29857009,0,S-116680003,900000000000011006,900000000000451002
2,102025,20020131,1,900000000000207008,S-10000006,S-9972008,0,S-116680003,900000000000011006,900000000000451002
8,108026,20220706,1,999000041000000102,S-272026007,S-25087005,2,S-363698007,900000000000011006,900000000000451002
14,114022,20020131,1,900000000000207008,S-134035007,S-84371003,0,S-116680003,900000000000011006,900000000000451002
27,127021,20020131,1,900000000000207008,S-134136005,S-57250008,0,S-116680003,900000000000011006,900000000000451002


In [54]:
active_relat[(active_relat.typeId == "S-370132008")].to_csv("snomed_codes_quantified.csv")
active_relat[(active_relat.typeId == "S-370132008")].destinationId.unique()

array(['S-26716007', 'S-30766002', 'S-117362005', 'S-117363000',
       'S-117365007'], dtype=object)

In [55]:
# write the relationship terms plus drug extension relationships to csv:
active_relat.to_csv(
    f"snomed_rela_csv_SNOMED-CT-active_UK_MONORelease_{release}_{date}.csv"
)  # snomed_rela_csv_SNOMED-CT-full_UK_drug_ext_Release_20210317

## Getting Quantitative/Ordinal Codes


In [56]:
scaled_codes = active_relat[(active_relat.typeId == "S-370132008")].sourceId.unique()
scaled_codes_relats = active_relat[active_relat.sourceId.isin(scaled_codes)]
scaled_codes_relats.typeId.unique()
relations_of_interest = [
    "S-370132008",  # Scale type
    "S-370130000",  # Proporty
    "S-704319004",  # Inheres-in
    "S-718497002",  # Inherent location
    "S-704321009",  # Charecterizes
    "S-704327008",  # Direct-Site
    "S-704324001",  # Process-output
    "S-246514001",  # Units
]
scaled_codes_relats = scaled_codes_relats[scaled_codes_relats.typeId.isin(relations_of_interest)]
scaled_codes_relats.to_csv("scaled_codes_relations.csv")

In [57]:
# # Find all types of relationships
# rel = all_active_relat['typeId'].unique()
# for _ in rel:
#     print(find_name(_), _)

### Parents and Children
Subtype relationship 116680003|Is a (attribute)| relates a Concept to its immediate supertype Concepts.

In [58]:
from collections import defaultdict

import pandas as pd


active_relat = pd.read_csv(f"snomed_rela_csv_SNOMED-CT-active_UK_MONORelease_{release}_{date}.csv")

pt2ch = defaultdict(list)
ch2pt = defaultdict(list)

for index, v in active_relat[active_relat.typeId == "S-116680003"].iterrows():
    # Parent to Children dictionary
    pt2ch[v["destinationId"]].append(v["sourceId"])

    # Children to Parent dictionary ("Is a" relationships)
    ch2pt[v["sourceId"]].append(v["destinationId"])

In [59]:
# Write to 'isa' relationships to file
with open(f"isa_active_rela_pt2ch_{release}_{date}.json", "w") as outfile:
    json.dump(dict(pt2ch), outfile)
with open(f"isa_active_rela_ch2pt_{release}_{date}.json", "w") as outfile:
    json.dump(dict(ch2pt), outfile)

In [60]:
# Load 'isa' relationships to df
with open(f"{drug_suffix}_isa_active_rela_pt2ch.json") as json_file:
    pt2ch = json.load(json_file)
with open(f"{drug_suffix}_isa_active_rela_ch2pt.json") as json_file:
    ch2pt = json.load(json_file)

NameError: name 'drug_suffix' is not defined

In [ ]:
ch2pt